# A Duel Became a Crowd — reproducible verification

This notebook **re-derives every headline number** in the blog from the orbital-launch
census (1957-2018, 5,726 attempts from Jonathan McDowell's JSR catalog, behind The
Economist's *The space race is dominated by new contenders*).

It is **download-and-run-local** (no Colab). The country-year matrix and the
success/attempt tables below are the Analyst's derived aggregates
(`code/derived_tables.json` / `code/client_data.json`); the full from-raw-CSV pipeline
is `code/analyze.py` over `launches.csv`. Every cell ends in an `assert`, so a green run
== the published figures reproduce. 2018 is a **partial** year (Jan-Oct).


## 1. The derived data (analyst aggregates)


In [ ]:
# Per-year launches by country (SU+RU merged into USSR/Russia; F+I-ESA+I-ELDO into Europe).
# columns: year, USSR/Russia, USA, China, Europe, Japan, India, Other
import math
MATRIX = [
  [1957,2,1,0,0,0,0,0],[1958,5,22,0,0,0,0,0],[1959,4,18,0,0,0,0,0],[1960,8,29,0,0,0,0,0],
  [1961,9,41,0,0,0,0,0],[1962,22,59,0,0,0,0,0],[1963,23,46,0,0,0,0,0],[1964,36,63,0,0,0,0,0],
  [1965,53,70,0,1,0,0,0],[1966,51,77,0,1,2,0,0],[1967,74,61,0,2,1,0,1],[1968,79,48,0,1,0,0,0],
  [1969,81,41,0,1,1,0,0],[1970,87,28,1,2,2,0,2],[1971,91,34,1,3,2,0,3],[1972,79,32,0,0,1,0,1],
  [1973,88,25,1,0,0,0,0],[1974,85,23,2,0,1,0,2],[1975,93,30,3,3,2,0,1],[1976,100,26,3,0,2,0,0],
  [1977,102,26,0,0,2,0,0],[1978,91,33,1,0,3,0,0],[1979,88,16,1,1,2,1,0],[1980,89,14,0,1,2,1,0],
  [1981,100,19,1,2,3,1,0],[1982,108,18,1,1,1,0,0],[1983,100,22,1,2,3,1,0],[1984,97,22,3,4,3,0,0],
  [1985,100,18,1,4,2,0,0],[1986,94,9,2,3,2,0,0],[1987,96,9,2,2,3,1,0],[1988,94,11,4,7,2,1,2],
  [1989,75,18,0,7,2,0,0],[1990,79,27,5,6,3,0,1],[1991,61,19,1,8,2,0,0],[1992,55,29,4,7,1,1,0],
  [1993,48,25,1,7,1,1,0],[1994,49,27,5,8,2,2,0],[1995,33,30,3,11,1,0,1],[1996,27,33,4,11,1,1,0],
  [1997,29,38,6,12,2,1,1],[1998,25,36,6,11,2,0,2],[1999,21,31,4,16,1,1,3],[2000,31,29,5,16,0,0,2],
  [2001,23,23,1,8,1,2,0],[2002,24,18,5,12,3,1,1],[2003,19,26,6,6,3,2,0],[2004,21,19,8,3,0,1,1],
  [2005,23,16,6,8,2,1,0],[2006,23,23,6,6,6,1,0],[2007,20,20,10,9,2,3,1],[2008,24,20,11,7,1,3,1],
  [2009,31,25,6,7,3,2,3],[2010,28,15,15,6,2,3,2],[2011,28,19,19,7,3,3,1],[2012,24,16,19,10,2,2,5],
  [2013,29,20,15,7,3,3,1],[2014,30,24,16,11,4,4,1],[2015,24,20,19,11,4,5,1],[2016,17,22,22,11,4,7,2],
  [2017,18,30,18,11,7,5,1],[2018,10,27,28,6,5,4,0],  # 2018 PARTIAL (Jan-Oct)
]
COLS = ['year','USSR/Russia','USA','China','Europe','Japan','India','Other']
def col(name): i = COLS.index(name); return [r[i] for r in MATRIX]
SUCCESS_BY_COUNTRY = {'USSR/Russia':(3024,3178),'USA':(1585,1716),'China':(289,302),
  'Europe':(294,307),'Japan':(106,115),'India':(58,65),'Other':(28,43)}
SUCCESS_BY_TYPE = {'state':(4462,4776),'private':(857,880),'startup':(65,70)}
def r1(x): return math.floor(x*10+0.5)/10
print('rows:', len(MATRIX), 'years', MATRIX[0][0], '-', MATRIX[-1][0])


## 2. The census: 5,726 attempts, 94.0% success


In [ ]:
successes, failures = 5384, 342
total = successes + failures
rate = round(100*successes/total, 1)
print(f'{total} attempts, {successes} successes -> {rate}% overall success')
assert total == 5726 and rate == 94.0


## 3. Volume is flat-to-down: peak 1967 = 139; 2018 partial = 80


In [ ]:
totals = [(r[0], sum(r[1:])) for r in MATRIX]
peak = max(totals, key=lambda t: t[1])
y2018 = dict(totals)[2018]
print('peak year:', peak, '| 2018 (partial):', y2018)
assert peak == (1967, 139) and y2018 == 80


## 4. The lead: the two-superpower share fell ~99% -> 54.5%


In [ ]:
def decade_rows(d0): return [r for r in MATRIX if d0 <= r[0] <= d0+9]
def duopoly(rows):
    top2 = sum(r[1]+r[2] for r in rows)   # USSR/Russia + USA
    return r1(100*top2/sum(sum(r[1:]) for r in rows))
s1960 = duopoly(decade_rows(1960)); s2010 = duopoly(decade_rows(2010))
# per-year endpoints the hero scrubber produces:
def year_duopoly(y): r = next(x for x in MATRIX if x[0]==y); return r1(100*(r[1]+r[2])/sum(r[1:]))
print('1960s', s1960, '%  2010s', s2010, '%  | per-year 1965', year_duopoly(1965), ' 2018', year_duopoly(2018))
assert s2010 == 54.5 and year_duopoly(1965) == 99.2 and year_duopoly(2018) == 46.3


## 5. China overtakes in 2018 (partial, annualized, and full-year)


In [ ]:
china_by_decade = {'1970s':13,'1980s':15,'1990s':39,'2000s':64,'2010s':171}
row2018 = next(r for r in MATRIX if r[0]==2018)
partial = {'China':row2018[3],'USA':row2018[2],'USSR/Russia':row2018[1]}
naive = {k: round(v*12/10,1) for k,v in partial.items()}
external = {'China':39,'USA':34,'USSR/Russia':20}
print('decade ramp', china_by_decade)
print('partial', partial, '| naive', naive, '| full-year', external)
assert partial == {'China':28,'USA':27,'USSR/Russia':10}
assert naive['China'] == 33.6 and china_by_decade['2010s'] == 171
assert partial['China'] > partial['USA'] and external['China'] > external['USA']


## 6. The old world still owns the ledger: 85.5%


In [ ]:
cum = {c: sum(col(c)) for c in COLS[1:]}
grand = sum(cum.values())
old = r1(100*(cum['USSR/Russia']+cum['USA'])/grand)
print({k: cum[k] for k in ['USSR/Russia','USA','Europe','China']}, '| old-world', old, '%')
assert cum['USSR/Russia']==3178 and cum['USA']==1716 and cum['Europe']==307 and cum['China']==302
assert old == 85.5 and cum['Europe'] > cum['China']


## 7. They are not cutting corners: China 95.7% ties the incumbents


In [ ]:
def rate(pair): return r1(100*pair[0]/pair[1])
by_country = {c: rate(p) for c,p in SUCCESS_BY_COUNTRY.items()}
by_type = {t: rate(p) for t,p in SUCCESS_BY_TYPE.items()}
print('by country', by_country)
print('by type', by_type)
assert by_country['China'] == 95.7 and by_type['private'] == 97.4


## 8. The two new contenders: China = 287/302 Long March; startup from 2006


In [ ]:
long_march, china_total = 287, 302
lm_share = r1(100*long_march/china_total)
startup = [('Falcon 9',62,2010),('Falcon 1',5,2006),('Electron',2,2017),('Falcon Heavy',1,2018)]
first_startup = min(y for _,_,y in startup)
startup_total = sum(n for _,n,_ in startup)
print('Long March share', lm_share, '% | first startup', first_startup, '| startup total', startup_total)
assert lm_share == 95.0 and first_startup == 2006 and startup_total == 70


---
**All assertions passed** — every published headline reproduces from the derived census.


In [ ]:
print('OK - all headline numbers reproduced.')
